In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

In [2]:
dataset = pd.read_csv('CKD.csv')

In [3]:
dataset=pd.get_dummies(dataset,drop_first=True)

In [4]:
indep=dataset[['age', 'bp', 'al', 'su', 'bgr', 'bu', 'sc', 'sod', 'pot', 'hrmo', 'pcv','wc', 'rc', 'sg_b', 'sg_c', 'sg_d', 'sg_e', 'rbc_normal', 'pc_normal','pcc_present', 'ba_present', 'htn_yes', 'dm_yes', 'cad_yes','appet_yes', 'pe_yes', 'ane_yes']]
dep=dataset['classification_yes']

In [5]:
#split into training set and test
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(indep, dep, test_size = 1/3, random_state = 0)


In [6]:
from sklearn.preprocessing import StandardScaler
sc = StandardScaler()
X_train = sc.fit_transform(X_train)
X_test = sc.transform(X_test)

In [7]:
from sklearn.svm import SVC

In [14]:
from sklearn.model_selection import GridSearchCV

param_grid = {'kernel':['linear','rbf','poly','sigmoid'],
             'gamma':['auto','scale'],
             'C':[10,100,1000,2000,3000]} 

grid = GridSearchCV(SVC(probability=True), param_grid,cv=5, refit = True, verbose = 3,n_jobs=-1,scoring='f1_weighted') 
   
grid.fit(X_train, y_train) 
 

Fitting 5 folds for each of 40 candidates, totalling 200 fits


[Parallel(n_jobs=-1)]: Using backend LokyBackend with 8 concurrent workers.
[Parallel(n_jobs=-1)]: Done  16 tasks      | elapsed:    0.0s
[Parallel(n_jobs=-1)]: Done 185 out of 200 | elapsed:    0.2s remaining:    0.0s
[Parallel(n_jobs=-1)]: Done 200 out of 200 | elapsed:    0.2s finished


GridSearchCV(cv=5, estimator=SVC(probability=True), n_jobs=-1,
             param_grid={'C': [10, 100, 1000, 2000, 3000],
                         'gamma': ['auto', 'scale'],
                         'kernel': ['linear', 'rbf', 'poly', 'sigmoid']},
             scoring='f1_weighted', verbose=3)

In [15]:
re=grid.cv_results_

grid_predictions = grid.predict(X_test) 
   
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(y_test, grid_predictions)
 
from sklearn.metrics import classification_report
clf_report = classification_report(y_test, grid_predictions)

In [16]:
from sklearn.metrics import f1_score
f1_macro=f1_score(y_test,grid_predictions,average='weighted')
print("The f1_macro value for best parameter {}:".format(grid.best_params_),f1_macro)

The f1_macro value for best parameter {'C': 10, 'gamma': 'auto', 'kernel': 'sigmoid'}: 0.9924946382275899


In [17]:
print("The confusion Matrix:\n",cm)

The confusion Matrix:
 [[51  0]
 [ 1 81]]


In [18]:
print("The report:\n",clf_report)

The report:
               precision    recall  f1-score   support

           0       0.98      1.00      0.99        51
           1       1.00      0.99      0.99        82

    accuracy                           0.99       133
   macro avg       0.99      0.99      0.99       133
weighted avg       0.99      0.99      0.99       133



In [19]:
from sklearn.metrics import roc_auc_score

roc_auc_score(y_test,grid.predict_proba(X_test)[:,1])


1.0

In [20]:
table=pd.DataFrame.from_dict(re)

In [21]:
table

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_C,param_gamma,param_kernel,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score
0,0.004788,0.000747,0.001994,1.762363e-06,10,auto,linear,"{'C': 10, 'gamma': 'auto', 'kernel': 'linear'}",0.981569,0.962636,0.962573,0.981031,0.981217,0.973805,0.009147,23
1,0.004602,0.000500,0.001404,4.972752e-04,10,auto,rbf,"{'C': 10, 'gamma': 'auto', 'kernel': 'rbf'}",0.963284,0.981014,1.000000,1.000000,1.000000,0.988859,0.014751,4
2,0.010172,0.000746,0.001796,3.990654e-04,10,auto,poly,"{'C': 10, 'gamma': 'auto', 'kernel': 'poly'}",1.000000,0.981014,1.000000,1.000000,0.981217,0.992446,0.009252,3
3,0.004587,0.000797,0.001795,3.985909e-04,10,auto,sigmoid,"{'C': 10, 'gamma': 'auto', 'kernel': 'sigmoid'}",0.981569,1.000000,1.000000,0.981031,1.000000,0.992520,0.009163,1
4,0.004189,0.000399,0.001995,6.312056e-04,10,scale,linear,"{'C': 10, 'gamma': 'scale', 'kernel': 'linear'}",0.981569,0.962636,0.962573,0.981031,0.981217,0.973805,0.009147,23
5,0.006383,0.001493,0.001995,9.725608e-07,10,scale,rbf,"{'C': 10, 'gamma': 'scale', 'kernel': 'rbf'}",0.963284,0.981014,1.000000,1.000000,1.000000,0.988859,0.014751,4
6,0.010770,0.000746,0.001796,3.989705e-04,10,scale,poly,"{'C': 10, 'gamma': 'scale', 'kernel': 'poly'}",1.000000,0.981014,1.000000,0.981031,0.981217,0.988652,0.009266,6
7,0.005187,0.000746,0.001794,3.984690e-04,10,scale,sigmoid,"{'C': 10, 'gamma': 'scale', 'kernel': 'sigmoid'}",0.981569,1.000000,1.000000,0.981031,1.000000,0.992520,0.009163,1
8,0.005386,0.002862,0.002194,1.465649e-03,100,auto,linear,"{'C': 100, 'gamma': 'auto', 'kernel': 'linear'}",0.981569,0.962636,0.962573,0.981031,0.981217,0.973805,0.009147,23
9,0.004588,0.000489,0.001995,6.304497e-04,100,auto,rbf,"{'C': 100, 'gamma': 'auto', 'kernel': 'rbf'}",0.963284,0.981014,1.000000,0.981031,1.000000,0.985066,0.013807,9


In [22]:
age_input=float(input("Age:"))
bp_input=float(input("bp:"))
al_input=float(input("al:"))
su_input=float(input("su:"))
bgr_input=float(input("bgr:"))
bu_input=float(input("bu:"))
sc_input=float(input("sc:"))
sod_input=float(input("sod:"))
pot_input=float(input("pot:"))
hrmo_input=float(input("hrmo:"))
pcv_input=float(input("pcv:"))
wc_input=float(input("wc:"))
rc_input=float(input("rc:"))
sg_b_input=float(input("sg_b yes 0 or 1:"))
sg_c_input=float(input("sg_c Yes 0 or 1:"))
sg_d_input=float(input("sg_d Yes 0 or 1:"))
sg_e_input=float(input("sg_e Yes 0 or 1:"))
rbc_normal_input=float(input("rbc Normal 0 or 1:"))
pc_normal_input=float(input("pc Normal 0 or 1:"))
pcc_present_input=float(input("pcc Present 0 or 1:"))
ba_present_input=float(input("ba Present 0 or 1:"))
htn_yes_input=float(input("htn Yes 0 or 1:"))
dm_yes_input=float(input("dm Yes 0 or 1:"))
cad_yes_input=float(input("cad Yes 0 or 1:"))
appet_yes_input=float(input("appet Yes 0 or 1:"))
pe_yes_input=float(input("pe Good 0 or 1:"))
ane_yes_input=float(input("ane Yes 0 or 1:"))

Age:22
bp:60
al:97
su:0
bgr:97
bu:18
sc:1
sod:134
pot:4
hrmo:13
pcv:42
wc:7900
rc:6
sg_b yes 0 or 1:1
sg_c Yes 0 or 1:0
sg_d Yes 0 or 1:0
sg_e Yes 0 or 1:0
rbc Normal 0 or 1:1
pc Normal 0 or 1:1
pcc Present 0 or 1:0
ba Present 0 or 1:0
htn Yes 0 or 1:0
dm Yes 0 or 1:0
cad Yes 0 or 1:0
appet Yes 0 or 1:1
pe Good 0 or 1:0
ane Yes 0 or 1:0


In [23]:
Future_Prediction=grid.predict([[age_input,bp_input,al_input,su_input,bgr_input,bu_input,sc_input,sod_input,pot_input,hrmo_input,pcv_input,wc_input,rc_input,sg_b_input,sg_c_input,sg_d_input,sg_e_input,rbc_normal_input,pc_normal_input,pcc_present_input,ba_present_input,htn_yes_input,dm_yes_input,cad_yes_input,appet_yes_input,pe_yes_input,ane_yes_input]])
print("Future_Prediction={}".format(Future_Prediction))

Future_Prediction=[0]
